# Benchmarking flow with real processor

https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-1099
https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-1221

This notebook demoes the execution of a real processor using the benchmarking flow, and the display of the generated artifacts.

The goal of this story is to run this flow from the github actions. But we still need this notebook to initialize its environment.

## Initialization

In [ ]:
# Access to Prefect
import os
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard}")

In [ ]:
# Choose prefect deployment method
from resources.widget_utils import *
deploy_prefect_radio

In [ ]:
# Choose prefect flow run method
run_prefect_radio

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *  
init_demo() # reload the global vars again
from resources.utils import *  

## Initialize the Prefect variables for mock responses

In [ ]:
# Read the json (copied from https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-1099)
with open("1221_benchmark_a_real_processing_flow.json", encoding="utf-8") as f:
    mock_template = json.load(f)

# Get all existing processor names
from rs_client.ogcapi.dpr_client import DprProcessor
processor_names = [p.value for p in DprProcessor]
print(f"Processor names: {processor_names}")

# Get all processor versions from the json
processor_versions = sorted(set(
    [v for s in mock_template["scenarios"].values() for v in s["processor_versions"].keys()]
))
print(f"Processor versions: {processor_versions}")

# Get all scenarios from the json
scenario_names = sorted(set(
    [s for s in mock_template["scenarios"].keys()]
))
print(f"Scenarios: {scenario_names}")

# Create one prefect variable for each processor with the json contents
from prefect.variables import Variable
for processor_name in processor_names:
    await Variable.set(
        name=f"benchmarking-{processor_name}-settings",
        value=mock_template, 
        tags=["processing", "all", "benchmarking"], 
        overwrite=True
    )

## Deploy the Prefect flow

In [ ]:
cadip_staging_deployed, benchmarking_deployed = await deploy_prefect(
    deploy_file="./1221_benchmark_a_real_processing_flow.yaml", 
    s3_code_folder=f"users/{OWNER_ID}/code", 
    work_pool_name=os.environ["PREFECT_WORK_POOL_SANDBOX"]
)

## Setup flow parameters

In [ ]:
processor_name = widgets.RadioButtons(
    options=processor_names,
    description="Processor:",
)
display(processor_name)

In [ ]:
processor_version = widgets.RadioButtons(
    options=processor_versions,
    description="Processor version:",
)
display(processor_version)

In [ ]:
scenario_name = widgets.RadioButtons(
    options=scenario_names,
    description="Scenario:",
)
display(scenario_name)

In [ ]:
flow_parameters = {
  "env": {
    "owner_id": OWNER_ID,
    "called_by": "notebooks/sprints/sprint41/1221_benchmark_a_real_processing_flow/1221_benchmark_a_real_processing_flow.ipynb" 
  },
  "processor_name": processor_name.value,
  "processor_version": processor_version.value,
  "scenario_name": scenario_name.value,
}
print(json.dumps(flow_parameters, indent=2))

## Create Dask cluster

Create a Dask cluster that is matching the one expected by the JSON file

In [ ]:
from resources.dask_clusters.dask_main_env import *

# Init the processor dask cluster
print(f"** Init Dask cluster for: {processor_name.value!r} **")
match processor_name.value:
    case "mockup":
        cluster_info_eopf = await init_dask_cluster_mockup()
    case DprProcessor.S1L0.value | DprProcessor.S3L0.value:
        cluster_info_eopf = await init_dask_cluster_l0()
    case DprProcessor.S1ARD.value:
        cluster_info_eopf = await init_dask_cluster_s1ard()
    case DprProcessor.S3L1OLCI.value | DprProcessor.S3L2OLCI.value:
        cluster_info_eopf = await init_dask_cluster_s3olci()

# Reload the global vars again
from resources.utils import *

## Stage the CADIP data

Stage the CADIP data expected by the JSON file to the correct collection

In [ ]:
from resources.widget_utils import *
from rs_workflows.cadip_flow import on_demand_cadip_staging

# Create collection to store benchamrking data
# The collection name is the same one as in the JSON file
CATALOG_COLLECTION_ID = "benchmarking-s3l0-input"
collection = create_test_collection(CATALOG_COLLECTION_ID)

In [ ]:
# Create staging cluster
await init_dask_cluster_staging()

# Prefect flow environment arguments
flow_env_args = {
  "env": {
    "owner_id": OWNER_ID,
    "logging_level": prefect_logging_level_radio.value,
  },
}

In [ ]:
# Data to stage before running the benchmark. It must match the data in the JSON file
session = {
    "cadip_collection_identifier": "cadip",
    "session_identifier": "S3A_20200121061417020456"
}

# Build params and run staging
params = {
    **flow_env_args,
    "cadip_collection_identifier": session["cadip_collection_identifier"],
    "session_identifier": session["session_identifier"],
    "catalog_collection_identifier": CATALOG_COLLECTION_ID,
}
await run_prefect(
    deploy_name=cadip_staging_deployed, 
    py_func=on_demand_cadip_staging, 
    params=params
)

# Check staged items in the catalog
staged = catalog_client.search(
    owner_id=OWNER_ID, 
    collections=[CATALOG_COLLECTION_ID]
).items
display(staged)
assert len(staged) == 1

## Run the Prefect flow

In [ ]:
from rs_workflows.benchmarking.benchmarking_flow import benchmark_processor
state = await run_prefect(
    deploy_name=benchmarking_deployed, 
    py_func=benchmark_processor, 
    params=flow_parameters
)

In [ ]:
# Read the artifact attached to the flow run ID
artifact = prefect_utils.read_artifact(
    http_session, state.state_details.flow_run_id, "benchmarking-result"
)

# Render the artifact as markdown
from IPython.display import Markdown
display(Markdown(artifact))